In [1]:
import pandas as pd

In [2]:
contig_annotations = pd.read_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/filtered_contig_annotations.csv")
#print(contig_annotations.head())

In [3]:
required_columns = ["barcode", "chain", "v_gene", "j_gene", "d_gene", "cdr3","umis","reads","raw_clonotype_id"]
if not all(col in contig_annotations.columns for col in required_columns):
    raise ValueError(f"Input data must contain the following columns: {required_columns}")


In [4]:
from collections import Counter
from itertools import combinations

Total_chains = len(contig_annotations)
print(f"\nTotal number of chains: {Total_chains}")


chain_counts = contig_annotations.groupby("barcode")["chain"].apply(list)
barcode_classes = chain_counts.apply(lambda x: "single_chain" if len(set(x)) == 1 and ("TRA" in x or "TRB" in x) 
                                     else "double_chain" if set(x) == {"TRA", "TRB"} and len(x) == 2 
                                     else "multiple_chains")
df_multi_chain = contig_annotations[contig_annotations["barcode"].isin(barcode_classes[barcode_classes == "multiple_chains"].index)]
unique_barcodes = contig_annotations["barcode"].nunique()
print(f"Unique barcodes: {unique_barcodes}")

multi_chain_combinations = df_multi_chain.groupby("barcode")["chain"].apply(lambda x: tuple(sorted(x)))
combination_counts = Counter(multi_chain_combinations)
total_barcodes = df_multi_chain["barcode"]
print("\nMultiple Chain Combinations:")
for combo, count in combination_counts.items():
    print(f"   {combo}: {count}")
print(f"\nTotal number of combination barcodes: {len(total_barcodes)}")
df_single_double = contig_annotations[contig_annotations["barcode"].isin(barcode_classes[barcode_classes.isin(["single_chain", "double_chain"])].index)]
df_multi_chain =contig_annotations[contig_annotations["barcode"].isin(barcode_classes[barcode_classes == "multiple_chains"].index)]
single_count = (barcode_classes == "single_chain").sum()
double_count = (barcode_classes == "double_chain").sum()
multi_count = (barcode_classes == "multiple_chains").sum()
total_count = len(df_multi_chain)
print("\nBarcode Category Breakdown:")
print(f"Single-chain barcodes: {single_count}")
print(f"Double-chain barcodes: {double_count}")
print(f"Multiple-chain barcodes: {multi_count}")

#print(f"Single-chain: {single_count}, Double-chain_barcode: {double_count}, Multiple-chain_unique_barcode: {multi_count}")

# computed_total = single_count + double_count + multi_count
# print(f"Total number of computed barcodes: {computed_total}")
# barcode_no = contig_annotations["barcode"].nunique() 
# if computed_total != barcode_no:
#     raise ValueError(f"Mismatch: Computed {computed_total}, but found {barcode_no} unique barcodes!")

# computed_total_chains = single_count + double_count*2 + total_count
# print(f"Total number of computed chains: {computed_total_chains}")

# if computed_total_chains != Total_barcode:
#     raise ValueError(f"Mismatch: Computed {computed_total_chains}, but found {Total_barcode}  barcodes!")
# Count how many single-chain barcodes are TRA (alpha) and TRB (beta)



Total number of chains: 9053
Unique barcodes: 4515

Multiple Chain Combinations:
   ('TRA', 'TRA', 'TRB'): 389
   ('TRA', 'TRB', 'TRB'): 180
   ('TRA', 'TRA', 'TRB', 'TRB'): 92

Total number of combination barcodes: 2075

Barcode Category Breakdown:
Single-chain barcodes: 730
Double-chain barcodes: 3124
Multiple-chain barcodes: 661


In [6]:
print(barcode_classes.value_counts())  # Show count of each class


chain
double_chain       3124
single_chain        730
multiple_chains     661
Name: count, dtype: int64


In [7]:
# Ensure 'd_gene' column exists and is valid
df_with_d_gene = contig_annotations[
    contig_annotations["d_gene"].notna() & (contig_annotations["d_gene"] != "")
]

# Count occurrences of each unique D gene (total chains)
d_gene_counts = df_with_d_gene["d_gene"].value_counts()

# Count unique barcodes associated with each D gene
d_gene_barcode_counts = df_with_d_gene.groupby("d_gene")["barcode"].nunique()

# Create a summary DataFrame
d_gene_summary_df = pd.DataFrame({
    "d_gene": d_gene_counts.index,
    "Total Chains": d_gene_counts.values,
    "Unique Barcodes": d_gene_barcode_counts.values
})

# Print the summary table in a readable format
print("\nd_gene Summary Table:")
print(d_gene_summary_df.to_string(index=False))

# Calculate total unique barcodes containing D genes
total_unique_barcodes = df_with_d_gene["barcode"].nunique()
print(f"\nTotal number of d_genes with unique barcodes: {total_unique_barcodes}")

# Calculate total chains with D genes
total_d_gene_chains = d_gene_counts.sum()
print(f"Total number of chains with d_genes: {total_d_gene_chains}")



d_gene Summary Table:
d_gene  Total Chains  Unique Barcodes
 TRBD1           795              788
 TRBD2           275              274

Total number of d_genes with unique barcodes: 1057
Total number of chains with d_genes: 1070


In [8]:
# Filter multi-chain entries with valid D genes
df_with_d_gene_multi = df_multi_chain[
    df_multi_chain["d_gene"].notna() & (df_multi_chain["d_gene"] != "")
]

# Count occurrences of each unique D gene in multi-chains
d_gene_multi_counts = df_with_d_gene_multi["d_gene"].value_counts()

# Count unique barcodes associated with each D gene in multi-chains
d_gene_barcode_counts_multi = df_with_d_gene_multi.groupby("d_gene")["barcode"].nunique()

# Create a DataFrame for summary
d_gene_summary_multi = pd.DataFrame({
    "d_Gene": d_gene_multi_counts.index,
    "Total Chains": d_gene_multi_counts.values,
    "Unique Barcodes": d_gene_barcode_counts_multi.values
})

# Print the summary table neatly
print("\nd_gene Distribution in Multi-Chains:")
print(d_gene_summary_multi.to_string(index=False))  

# Total count of D genes in multi-chains
d_gene_multi_total = d_gene_multi_counts.sum()
print(f"\nTotal number of chains with D genes in multi-chains: {d_gene_multi_total}")


# Filter single & double chain entries with valid D genes
df_with_d_gene_single = df_single_double[
    df_single_double["d_gene"].notna() & (df_single_double["d_gene"] != "")
]

# Count occurrences of each unique D gene in single & double chains
d_gene_single_counts = df_with_d_gene_single["d_gene"].value_counts()

# Print formatted D Gene counts for single & double chains
print("\nd_gene Counts in Single & Double Chains:")
for gene, count in d_gene_single_counts.items():
    print(f"   {gene}: {count}")

# Get total number of chains with D genes in single and double chains
d_gene_single_total = d_gene_single_counts.sum()
print(f"\nTotal number of chains with D genes in single and double chains: {d_gene_single_total}")




d_gene Distribution in Multi-Chains:
d_Gene  Total Chains  Unique Barcodes
 TRBD1           149              142
 TRBD2            57               56

Total number of chains with D genes in multi-chains: 206

d_gene Counts in Single & Double Chains:
   TRBD1: 646
   TRBD2: 218

Total number of chains with D genes in single and double chains: 864


In [5]:
# Group by barcode to collect chains per cell
grouped = contig_annotations.groupby("barcode")


In [6]:
def safe_str(value):
    """Convert value to string, ensuring None values are handled safely."""
    return str(value) if pd.notna(value) and value is not None else "" 

clonotypes = []
alpha_map = {}  # Stores TRA clonotypes
beta_map = {}   # Stores TRB clonotypes
combined_map = {}  # Stores combined TRA-TRB clonotypes

for barcode, group in grouped:
    chains = {"TRA": [], "TRB": []}

    for _, row in group.iterrows():
        if row["chain"].upper() in chains:
            chains[row["chain"].upper()].append(row)

    # Store all possible clonotypes
    combined_clonotypes = set()
    
    raw_clonotype_id = contig_annotations.loc[
        contig_annotations["barcode"] == barcode, "raw_clonotype_id"
    ].values[0] if not contig_annotations.loc[
        contig_annotations["barcode"] == barcode, "raw_clonotype_id"
    ].empty else None
    
    # Pair every TRA with every TRB
    if chains["TRA"] and chains["TRB"]:
        for tra in chains["TRA"]:
            alpha_key = (safe_str(tra["v_gene"]), safe_str(tra["j_gene"]), safe_str(tra["cdr3"]))
            raw_clonotype_alpha = alpha_map.get(alpha_key, "_".join(filter(None, alpha_key)))
            alpha_map[alpha_key] = raw_clonotype_alpha  # Store in map

            for trb in chains["TRB"]:
                beta_key = (safe_str(trb["v_gene"]), safe_str(trb["j_gene"]), safe_str(trb["cdr3"]))
                raw_clonotype_beta = beta_map.get(beta_key, "_".join(filter(None, beta_key)))
                beta_map[beta_key] = raw_clonotype_beta  # Store in map

                # Assign a unique combined clonotype
                combined_key = (raw_clonotype_alpha, raw_clonotype_beta)
                if combined_key not in combined_map:
                    combined_map[combined_key] = f"{raw_clonotype_alpha}_{raw_clonotype_beta}"
                combined_clonotype = combined_map[combined_key]

                combined_clonotypes.add(combined_clonotype)

                # Append each TRA-TRB pair as a separate row
                clonotypes.append({
                    "barcode": barcode,
                    "v_gene_alpha": tra["v_gene"],
                    "j_gene_alpha": tra["j_gene"],
                    "cdr3_alpha": tra["cdr3"],
                    "raw_clonotype_alpha": raw_clonotype_alpha,
                    "v_gene_beta": trb["v_gene"],
                    "j_gene_beta": trb["j_gene"],
                    "cdr3_beta": trb["cdr3"],
                    "raw_clonotype_beta": raw_clonotype_beta,
                    "combined_raw_clonotype": combined_clonotype,
                    "umi_alpha": tra["umis"],  # 🔹 Keep individual umi/read values
                    "read_alpha": tra["reads"],
                    "umi_beta": trb["umis"],
                    "read_beta": trb["reads"],
                    "raw_clonotype_id": raw_clonotype_id
                })

    # If only TRA exists, store those separately
    elif chains["TRA"]:
        for tra in chains["TRA"]:
            alpha_key = (safe_str(tra["v_gene"]), safe_str(tra["j_gene"]), safe_str(tra["cdr3"]))
            raw_clonotype_alpha = alpha_map.get(alpha_key, "_".join(filter(None, alpha_key)))
            alpha_map[alpha_key] = raw_clonotype_alpha  # Store in map

            clonotypes.append({
                "barcode": barcode,
                "v_gene_alpha": tra["v_gene"],
                "j_gene_alpha": tra["j_gene"],
                "cdr3_alpha": tra["cdr3"],
                "raw_clonotype_alpha": raw_clonotype_alpha,
                "v_gene_beta": None,
                "j_gene_beta": None,
                "d_gene_beta": None,
                "cdr3_beta": None,
                "raw_clonotype_beta": None,
                "combined_raw_clonotype": raw_clonotype_alpha,
                "umi_alpha": tra["umis"],  # 🔹 Keep individual umi/read values
                "read_alpha": tra["reads"],
                "umi_beta": None,
                "read_beta": None,
                "raw_clonotype_id": raw_clonotype_id
            })

    # If only TRB exists, store those separately
    elif chains["TRB"]:
        for trb in chains["TRB"]:
            beta_key = (safe_str(trb["v_gene"]), safe_str(trb["d_gene"]), safe_str(trb["j_gene"]), safe_str(trb["cdr3"]))
            raw_clonotype_beta = beta_map.get(beta_key, "_".join(filter(None, beta_key)))
            beta_map[beta_key] = raw_clonotype_beta  # Store in map

            clonotypes.append({
                "barcode": barcode,
                "v_gene_alpha": None,
                "j_gene_alpha": None,
                "cdr3_alpha": None,
                "raw_clonotype_alpha": None,
                "v_gene_beta": trb["v_gene"],
                "j_gene_beta": trb["j_gene"],
                "cdr3_beta": trb["cdr3"],
                "raw_clonotype_beta": raw_clonotype_beta,
                "combined_raw_clonotype": raw_clonotype_beta,
                "umi_alpha": None,
                "read_alpha": None,
                "umi_beta": trb["umis"],
                "read_beta": trb["reads"],
                "raw_clonotype_id": raw_clonotype_id
            })

# Convert to DataFrame
clonotype_df = pd.DataFrame(clonotypes)

print("✅ Clonotype data saved successfully!")


✅ Clonotype data saved successfully!


In [7]:
 clonotypes.append({
        "barcode": barcode,
        "v_gene_alpha": tra["v_gene"] if chains["TRA"] else None,
        "j_gene_alpha": tra["j_gene"] if chains["TRA"] else None,
        "cdr3_alpha": tra["cdr3"] if chains["TRA"] else None,
        "raw_clonotype_alpha": raw_clonotype_alpha,
        "v_gene_beta": trb["v_gene"] if chains["TRB"] else None,
        "j_gene_beta": trb["j_gene"] if chains["TRB"] else None,
        
        "cdr3_beta": trb["cdr3"] if chains["TRB"] else None,
        "raw_clonotype_beta": raw_clonotype_beta,
        "combined_raw_clonotype": combined_clonotypes,
        "umi_alpha": tra["umis"] if chains["TRA"] else None, 
        "read_alpha": tra["reads"] if chains["TRA"] else None,
        "umi_beta": trb["umis"] if chains["TRB"] else None,
        "read_beta": trb["reads"] if chains["TRB"] else None,
        "raw_clonotype_id": raw_clonotype_id
    })

In [8]:
clonotype_df.to_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/clonotype.csv", index=False)

In [49]:

# Ensure clonotype_id column exists
if "clonotype_id" not in clonotype_df.columns:
    clonotype_df["clonotype_id"] = pd.NA  

# Step 1: Assign clonotype IDs to FULL-CHAIN clonotypes
clonotype_df["combined_raw_clonotype"] = clonotype_df.apply(
    lambda row: "_".join(sorted(filter(pd.notna, [row["raw_clonotype_alpha"], row["raw_clonotype_beta"]])))
    if pd.notna(row["raw_clonotype_alpha"]) and pd.notna(row["raw_clonotype_beta"]) else pd.NA,
    axis=1
)

# Get unique full-chain clonotypes sorted by frequency
full_chain_clonotypes = clonotype_df.dropna(subset=["combined_raw_clonotype"]).copy()
clonotype_counts = full_chain_clonotypes["combined_raw_clonotype"].value_counts().reset_index()
clonotype_counts.columns = ["combined_raw_clonotype", "count"]
clonotype_counts["clonotype_id"] = [f"clonotype{i+1}" for i in range(len(clonotype_counts))]

# Merge clonotype IDs back into the dataframe
if "clonotype_id" in clonotype_df.columns:
    clonotype_df.drop(columns=["clonotype_id"], inplace=True)

clonotype_df = clonotype_df.merge(clonotype_counts[["combined_raw_clonotype", "clonotype_id"]], 
                                  on="combined_raw_clonotype", how="left")

# Step 2: Assign clonotype IDs to SINGLE-CHAIN (TRA or TRB only)
clonotype_id_start = len(clonotype_counts) + 1  # Continue numbering new clonotypes

# Create dictionaries to store mappings of single-chain clonotypes
alpha_map = {}
beta_map = {}

for index, row in clonotype_df.iterrows():
    if pd.notna(row["clonotype_id"]):  # Store full-chain clonotype IDs in lookup dictionaries
        if pd.notna(row["raw_clonotype_alpha"]):
            alpha_map[row["raw_clonotype_alpha"]] = row["clonotype_id"]
        if pd.notna(row["raw_clonotype_beta"]):
            beta_map[row["raw_clonotype_beta"]] = row["clonotype_id"]

# Assign clonotypes for single-chain TRA or TRB
for index, row in clonotype_df.iterrows():
    if pd.isna(row["clonotype_id"]):  # Process only missing IDs
        matched_id = None

        # Check for existing matches in single-chain clonotype mappings
        if pd.notna(row["raw_clonotype_alpha"]) and row["raw_clonotype_alpha"] in alpha_map:
            matched_id = alpha_map[row["raw_clonotype_alpha"]]
        elif pd.notna(row["raw_clonotype_beta"]) and row["raw_clonotype_beta"] in beta_map:
            matched_id = beta_map[row["raw_clonotype_beta"]]

        # Assign the matched clonotype ID if found, else create a new one
        if matched_id:
            clonotype_df.at[index, "clonotype_id"] = matched_id
        else:
            clonotype_df.at[index, "clonotype_id"] = f"clonotype{clonotype_id_start}"
            if pd.notna(row["raw_clonotype_alpha"]):
                alpha_map[row["raw_clonotype_alpha"]] = clonotype_df.at[index, "clonotype_id"]
            if pd.notna(row["raw_clonotype_beta"]):
                beta_map[row["raw_clonotype_beta"]] = clonotype_df.at[index, "clonotype_id"]
            clonotype_id_start += 1  # Increment for next new clonotype
# Compute frequency of each clonotype
clonotype_counts = clonotype_df["clonotype_id"].value_counts().reset_index()
clonotype_counts.columns = ["clonotype_id", "frequency"]
if "frequency" in clonotype_df.columns:
    clonotype_df.drop(columns=["frequency"], inplace=True)
clonotype_df = clonotype_df.merge(clonotype_counts, on="clonotype_id", how="left")

# Final Output
print("✅ Clonotype assignment completed successfully!")
#print(clonotype_df[["barcode", "raw_clonotype_alpha", "raw_clonotype_beta", "clonotype_id"]].drop_duplicates().head())
clonotype_df['step_number'] = 0

✅ Clonotype assignment completed successfully!


In [10]:
clonotype_df.to_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/clonotype_results.csv", index=False)

In [12]:
# Count cases where ONLY Alpha exists (Beta is NaN)
only_alpha = clonotype_df[(clonotype_df["raw_clonotype_alpha"].notna()) & (clonotype_df["raw_clonotype_beta"].isna())].shape[0]

# Count cases where ONLY Beta exists (Alpha is NaN)
only_beta = clonotype_df[(clonotype_df["raw_clonotype_beta"].notna()) & (clonotype_df["raw_clonotype_alpha"].isna())].shape[0]

print(f"✅ Total unique TRA (Alpha only): {only_alpha}")
print(f"✅ Total unique TRB (Beta only): {only_beta}")


✅ Total unique TRA (Alpha only): 54
✅ Total unique TRB (Beta only): 676


In [50]:
# Ensure no accidental spaces or hidden characters
clonotype_df["barcode"] = clonotype_df["barcode"].astype(str).str.strip()

# Identify duplicate barcodes (barcodes appearing more than once)
duplicate_barcodes = clonotype_df[clonotype_df.duplicated(subset=["barcode"], keep=False)].copy()

# Count occurrences of each barcode
barcode_counts = duplicate_barcodes["barcode"].value_counts()

# Filter only barcodes that appear more than once
duplicate_barcodes["barcode_repeat_count"] = duplicate_barcodes["barcode"].map(barcode_counts)

# Save all duplicate barcodes
duplicate_barcodes[["barcode", "clonotype_id", "frequency", "barcode_repeat_count",
                    "combined_raw_clonotype", "umi_alpha", "read_alpha", 
                    "umi_beta", "read_beta"]].to_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/duplicated_barcodes.csv", index=False)
print("✅ Duplicated barcodes are successfully segregated.")

# Calculate statistics
num_unique_duplicate_barcodes = duplicate_barcodes["barcode"].nunique()
num_total_duplicate_entries = len(duplicate_barcodes)

print(f"✅ Total unique duplicated barcodes: {num_unique_duplicate_barcodes}")
print(f"✅ Total occurrences of duplicated barcodes: {num_total_duplicate_entries}")

clonotype_df.update(duplicate_barcodes)
clonotype_df['step_number'] = 1

✅ Duplicated barcodes are successfully segregated.
✅ Total unique duplicated barcodes: 661
✅ Total occurrences of duplicated barcodes: 1506


In [14]:
num_clonotypes_shared = duplicate_barcodes["clonotype_id"].nunique()  # Unique clonotypes among duplicate barcodes

print(f"✅ Total different clonotypes assigned to duplicate barcodes: {num_clonotypes_shared}")
num_unique_clonotypes_freq_1 = duplicate_barcodes.loc[duplicate_barcodes["frequency"] == 1, "clonotype_id"].nunique()
print(num_unique_clonotypes_freq_1)


✅ Total different clonotypes assigned to duplicate barcodes: 1490
1462


In [15]:
df_triplets=duplicate_barcodes[duplicate_barcodes["barcode_repeat_count"]==2]
df_triplets[["barcode", "clonotype_id", "frequency", "barcode_repeat_count","combined_raw_clonotype","umi_alpha","read_alpha","umi_beta","read_beta"]].to_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/triplets.csv", index=False)
print(f"triplets are segregated successfully ")
df=duplicate_barcodes[duplicate_barcodes["barcode_repeat_count"]==4]
df[["barcode", "clonotype_id", "frequency", "barcode_repeat_count","combined_raw_clonotype","umi_alpha","read_alpha","umi_beta","read_beta"]].to_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/Quadruplets.csv", index=False)
print(f"Quadruplets are segregated successfully ")

num_clonotypes_shared = df_triplets["clonotype_id"].nunique()
print(f"✅ Total different clonotypes assigned to triplets barcodes: {num_clonotypes_shared}")
num_barcode_shared = df_triplets["barcode"].nunique()
print(f"✅ Total different unique barcodes in triplets: {num_barcode_shared}")


num_clonotypes_shared_4= df["clonotype_id"].nunique()
print(f"✅ Total different clonotypes assigned to Quadruplets barcodes: {num_clonotypes_shared_4}")


num_barcode_shared_4 = df["barcode"].nunique()
print(f"✅ Total different unique barcodes in Quadruplets: {num_barcode_shared_4}")

triplets are segregated successfully 
Quadruplets are segregated successfully 
✅ Total different clonotypes assigned to triplets barcodes: 1130
✅ Total different unique barcodes in triplets: 569
✅ Total different clonotypes assigned to Quadruplets barcodes: 366
✅ Total different unique barcodes in Quadruplets: 92


In [20]:
# Sort DataFrame by barcode, then clonotype_id (ascending), and frequency (descending)
df_sorted_triplets = df_triplets.sort_values(by=["barcode", "clonotype_id", "frequency"], ascending=[True, True, False])

# Function to filter clonotypes
def filter_clonotypes(group):
    # If all clonotypes for a barcode have frequency = 1, keep only the first one
    if (group["frequency"] == 1).all():
        return group.iloc[:1]  # Keep the first one

    # Otherwise, remove rows where frequency = 1 and keep the rest
    return group[group["frequency"] > 1]

# Apply the filtering function
df_filtered_triplets = df_sorted_triplets.groupby("barcode", group_keys=False).apply(filter_clonotypes)


# Save the filtered data to CSV
output_path = "/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/filtered_triplets.csv"
df_filtered_triplets[["barcode","combined_raw_clonotype","clonotype_id","frequency","umi_alpha","read_alpha","umi_beta","read_beta"]].to_csv(output_path, index=False)

print("Before filtering:", df_sorted_triplets["barcode"].nunique(), df_sorted_triplets["clonotype_id"].nunique())
print("After filtering:", df_filtered_triplets["barcode"].nunique(), df_filtered_triplets["clonotype_id"].nunique())


Before filtering: 569 1130
After filtering: 569 569


/tmp/ipykernel_315883/3680269238.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_filtered_triplets = df_sorted_triplets.groupby("barcode", group_keys=False).apply(filter_clonotypes)


In [21]:

# Sort DataFrame by barcode and frequency (highest first)
df_sorted = df.sort_values(by=["barcode", "clonotype_id","frequency"], ascending=[True,True, False])

# Function to filter clonotypes
def filter_clonotypes(group):
    # If all clonotypes for a barcode have frequency = 1, keep only the first one
    if (group["frequency"] == 1).all():
        return group.iloc[:1]  # Keep the first one

    # Otherwise, remove rows where frequency = 1 and keep the rest
    return group[group["frequency"] > 1]

# Apply the filtering function
df_filtered = df_sorted.groupby("barcode", group_keys=False).apply(filter_clonotypes)

# Save filtered data to CSV
output_path = "/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/filtered_quadruplets.csv"
df_filtered[["barcode","combined_raw_clonotype","clonotype_id","frequency","umi_alpha","read_alpha","umi_beta","read_beta"]].to_csv(output_path, index=False)

print("Before filtering duplicates:", df_sorted["barcode"].nunique(), df_sorted["clonotype_id"].nunique())
print("After filtering duplicates:", df_filtered["barcode"].nunique(), df_filtered["clonotype_id"].nunique())

Before filtering duplicates: 92 366
After filtering duplicates: 92 95


/tmp/ipykernel_315883/2702543945.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_filtered = df_sorted.groupby("barcode", group_keys=False).apply(filter_clonotypes)


In [51]:

# Sort DataFrame by barcode and frequency (highest first)
df_duplicated = duplicate_barcodes.sort_values(by=["barcode", "clonotype_id","frequency"], ascending=[True,True, False])

# Function to filter clonotypes
def filter_clonotypes(group):
    # If all clonotypes for a barcode have frequency = 1, keep only the first one
    if (group["frequency"] == 1).all():
        return group.iloc[:1]  # Keep the first one

    # Otherwise, remove rows where frequency = 1 and keep the rest
    return group[group["frequency"] > 1]
    
clonotype_df.update(df_duplicated)
clonotype_df['step_number'] = 2 
# Apply the filtering function
df_filtered_duplicate = df_duplicated.groupby("barcode", group_keys=False).apply(filter_clonotypes)

# Save filtered data to CSV
output_path = "/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/filtered_duplicates.csv"
df_filtered_duplicate[["barcode","combined_raw_clonotype","clonotype_id","frequency","umi_alpha","read_alpha","umi_beta","read_beta"]].to_csv(output_path, index=False)

# Calculate and print statistics
num_clonotypes_shared_duplicated = df_filtered_duplicate["clonotype_id"].nunique()
print(f"✅ Total different clonotypes assigned to filtered duplicate barcodes: {num_clonotypes_shared_duplicated}")

num_barcode_shared_duplicated = df_filtered_duplicate["barcode"].nunique()
print(f"✅ Total different unique barcodes in filtered duplicates : {num_barcode_shared_duplicated}")
clonotype_df.update(df_filtered_duplicate)
clonotype_df['step_number'] = 3 

✅ Total different clonotypes assigned to filtered duplicate barcodes: 658
✅ Total different unique barcodes in filtered duplicates : 661


/tmp/ipykernel_467597/2895899003.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_filtered_duplicate = df_duplicated.groupby("barcode", group_keys=False).apply(filter_clonotypes)


In [17]:

# # Sort DataFrame by barcode and frequency (highest first)
# df_filter = clonotype_df.sort_values(by=["barcode", "clonotype_id","frequency"], ascending=[True,True, False])

# # Function to filter clonotypes
# def filter_clonotypes(group):
#     # If all clonotypes for a barcode have frequency = 1, keep only the first one
#     if (group["frequency"] == 1).all():
#         return group.iloc[:1]  # Keep the first one

#     # Otherwise, remove rows where frequency = 1 and keep the rest
#     return group[group["frequency"] > 1]

# # Apply the filtering function
# df_filter = clonotype_df.groupby("barcode", group_keys=False).apply(filter_clonotypes)

# # Save filtered data to CSV
# output_path = "/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/filtered_clonotype.csv"
# df_filter[["barcode","combined_raw_clonotype","clonotype_id","frequency","umi_alpha","read_alpha","umi_beta","read_beta"]].to_csv(output_path, index=False)

# # Calculate and print statistics
# num_clonotypes = df_filter["clonotype_id"].nunique()
# print(f"✅ Total different clonotypes assigned to filtered duplicate barcodes: {num_clonotypes}")

# num_barcode = df_filter["barcode"].nunique()
# print(f"✅ Total different unique barcodes in filtered duplicates : {num_barcode}")


In [52]:
duplicate_left = df_filtered_duplicate[df_filtered_duplicate.duplicated(subset=["barcode"], keep=False)].copy()
# Count occurrences of each barcode
barcode_counts = duplicate_left["barcode"].value_counts()

# Filter only barcodes that appear more than once
duplicate_left["barcode_repeat_count"] = duplicate_left["barcode"].map(barcode_counts)

duplicate_left[["barcode", "clonotype_id", "frequency", "barcode_repeat_count",
                    "combined_raw_clonotype", "umi_alpha", "read_alpha", 
                    "umi_beta", "read_beta"]].to_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/duplicate_left.csv", index=False)
print("✅ left duplicates are saved as duplicate_left.csv")

# Calculate statistics
num_unique_duplicate_left = duplicate_left["barcode"].nunique()
num_unique_clonotype_left = duplicate_left["clonotype_id"].nunique()
num_total_duplicate_left = len(duplicate_left)

print(f"✅ Total unique duplicated barcodes: {num_unique_duplicate_left}")
print(f"✅ Total unique clonotypes: {num_unique_clonotype_left}")
print(f"✅ Total occurrences of duplicated barcodes: {num_total_duplicate_left}")
clonotype_df.update(duplicate_left)
clonotype_df['step_number'] = 4   

✅ left duplicates are saved as duplicate_left.csv
✅ Total unique duplicated barcodes: 13
✅ Total unique clonotypes: 14
✅ Total occurrences of duplicated barcodes: 26


In [20]:
pip install networkx

  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [31]:
import networkx as nx

# Step 1: Identify barcodes that appear exactly twice
barcode_counts = duplicate_left['barcode'].value_counts()
valid_barcodes = barcode_counts[barcode_counts == 2].index  # Only keep barcodes appearing exactly twice

# Step 2: Keep only barcodes appearing exactly twice
filtered_data = duplicate_left[duplicate_left['barcode'].isin(valid_barcodes)].copy()

# Step 3: Ensure the 'frequency' column is exactly 2 for BOTH occurrences of a barcode
valid_frequencies = (
    filtered_data.groupby('barcode')['frequency']
    .nunique()  # Check how many unique frequency values exist per barcode
    .eq(1) &  # Ensure only one unique frequency per barcode (so no mix like 2 & 8)
    filtered_data.groupby('barcode')['frequency'].first().eq(2)  # Ensure that frequency = 2
)

# Step 4: Keep only barcodes where BOTH occurrences have frequency = 2
valid_barcodes = valid_frequencies[valid_frequencies].index
filtered_data = filtered_data[filtered_data['barcode'].isin(valid_barcodes)]

# Step 5: Ensure each clonotype appears exactly twice
clonotype_counts = filtered_data["clonotype_id"].value_counts()
valid_clonotypes = clonotype_counts[clonotype_counts == 2].index
filtered_data = filtered_data[filtered_data["clonotype_id"].isin(valid_clonotypes)]

# Step 6: Double-check that each barcode still appears exactly twice
barcode_occurrences = filtered_data['barcode'].value_counts()
incorrect_barcodes = barcode_occurrences[barcode_occurrences != 2]

if not incorrect_barcodes.empty:
    print(f"❌ Warning: Some barcodes do not appear exactly twice!\n{incorrect_barcodes}")
else:
    print("✅ All selected barcodes appear exactly twice before merging.")

# Step 7: Group clonotypes by barcode
barcode_clonotypes = filtered_data.groupby('barcode')['clonotype_id'].apply(set).reset_index()

# Step 8: Create an undirected graph where edges exist if a barcode appears in multiple clonotypes
G = nx.Graph()

for _, row in barcode_clonotypes.iterrows():
    clonotypes = list(row['clonotype_id'])
    for i in range(len(clonotypes)):
        for j in range(i + 1, len(clonotypes)):
            G.add_edge(clonotypes[i], clonotypes[j])  # Connect clonotypes if they share a barcode

# Step 9: Find connected components (merged clonotype groups)
clonotype_mapping = {}
for component in nx.connected_components(G):
    min_clonotype = min(component)  # Choose the smallest clonotype ID
    for clonotype in component:
        clonotype_mapping[clonotype] = min_clonotype

# Step 10: Apply the mapping to update clonotype assignments
filtered_data['merged_clonotype'] = filtered_data['clonotype_id'].map(clonotype_mapping)
filtered_data['merged_clonotype'].fillna(filtered_data['clonotype_id'], inplace=True)  # Ensure no missing mappings

# Step 11: Verify that every unique barcode still appears exactly twice
barcode_occurrences_after = filtered_data['barcode'].value_counts()
incorrect_barcodes_after = barcode_occurrences_after[barcode_occurrences_after != 2]

if not incorrect_barcodes_after.empty:
    print(f"❌ Warning: Some barcodes do not appear exactly twice after merging!\n{incorrect_barcodes_after}")
else:
    print("✅ All barcodes still appear exactly twice after merging.")

# Step 12: Set frequency explicitly to 2 for all rows
filtered_data['frequency'] = 2

# Step 13: Add barcode repeat count (each barcode appears twice)
filtered_data['barcode_repeat_count'] = filtered_data.groupby('barcode')['barcode'].transform('count')

# Step 14: Debugging - Verify barcode counts
print(f"Unique barcodes before merging: {duplicate_left['barcode'].nunique()} ")
print(f"Total barcode occurrences before merging: {len(duplicate_left)}" )

merged_barcode_count = filtered_data['barcode'].nunique()
merged_total_barcodes = len(filtered_data)

print(f"Unique barcodes after filtering: {merged_barcode_count}") 
print(f"Total barcode occurrences after filtering: {merged_total_barcodes}")

# Step 15: Check if any barcodes appear only once or more than twice after merging (should be 0)
if len(incorrect_barcodes_after) > 0:
    print(f"❌ Warning: Some barcodes appear incorrectly after merging:\n{incorrect_barcodes_after}")
else:
    print("✅ No barcode appears more than twice or less than twice!")

clonotype_count = filtered_data["merged_clonotype"].nunique()
print(f"Unique merged clonotypes: {clonotype_count}")

barcode_count = filtered_data["barcode"].nunique()
print(f"Unique barcodes: {barcode_count}")

clonotype_frequency = filtered_data["merged_clonotype"].value_counts()
print("Clonotype Frequency Distribution:")
print(clonotype_frequency)

# Step 16: Save final result
output_path = "/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/matching_cells.csv"
filtered_data[["barcode", "clonotype_id", "frequency", "merged_clonotype", "barcode_repeat_count"]].to_csv(output_path, index=False)

print(f"✅ Saved {len(filtered_data)} matching cells to '{output_path}'")


✅ All selected barcodes appear exactly twice before merging.
✅ All barcodes still appear exactly twice after merging.
Unique barcodes before merging: 13 
Total barcode occurrences before merging: 26
Unique barcodes after filtering: 10
Total barcode occurrences after filtering: 20
✅ No barcode appears more than twice or less than twice!
Unique merged clonotypes: 5
Unique barcodes: 10
Clonotype Frequency Distribution:
merged_clonotype
clonotype30    4
clonotype15    4
clonotype56    4
clonotype5     4
clonotype14    4
Name: count, dtype: int64
✅ Saved 20 matching cells to '/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/matching_cells.csv'


/tmp/ipykernel_467597/805092114.py:57: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  filtered_data['merged_clonotype'].fillna(filtered_data['clonotype_id'], inplace=True)  # Ensure no missing mappings


In [32]:
excluded_data = duplicate_left[~duplicate_left['barcode'].isin(filtered_data['barcode'])]


In [33]:
import pandas as pd
import networkx as nx

# Step 1: Filter barcodes appearing exactly four times
barcode_counts = excluded_data ['barcode'].value_counts()
valid_barcodes = barcode_counts[barcode_counts == 4].index
filtered_data = excluded_data [duplicate_left['barcode'].isin(valid_barcodes)].copy()

# Step 2: Ensure ALL four occurrences have frequency = 4
valid_frequencies = (
    filtered_data.groupby('barcode')['frequency'].nunique().eq(1) &
    filtered_data.groupby('barcode')['frequency'].first().eq(2)
)
filtered_data = filtered_data[filtered_data['barcode'].isin(valid_frequencies[valid_frequencies].index)]

# Step 3: Ensure each clonotype appears exactly four times
clonotype_counts = filtered_data["clonotype_id"].value_counts()
valid_clonotypes = clonotype_counts[clonotype_counts == 2].index
filtered_data = filtered_data[filtered_data["clonotype_id"].isin(valid_clonotypes)]

# Step 4: Merge clonotypes using NetworkX
barcode_clonotypes = filtered_data.groupby('barcode')['clonotype_id'].apply(set).reset_index()

G = nx.Graph()
for _, row in barcode_clonotypes.iterrows():
    clonotypes = list(row['clonotype_id'])
    for i in range(len(clonotypes)):
        for j in range(i + 1, len(clonotypes)):
            G.add_edge(clonotypes[i], clonotypes[j])  # Correctly create edges between clonotypes

# Step 5: Assign merged clonotypes to the smallest clonotype ID
clonotype_mapping = {}
for component in nx.connected_components(G):
    min_clonotype = min(component)  # Smallest clonotype ID in the component
    for clonotype in component:
        clonotype_mapping[clonotype] = min_clonotype

# Step 6: Apply the mapping
filtered_data['merged_clonotype'] = filtered_data['clonotype_id'].map(clonotype_mapping).fillna(filtered_data['clonotype_id'])


# Step 7: Final checks and output
#filtered_data['frequency'] = 2
#filtered_data['barcode_repeat_count'] = filtered_data.groupby('barcode')['barcode'].transform('count')
filtered_barcode= filtered_data["barcode"].nunique()
print(filtered_barcode)
output_path = "/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/matching_cells_4x.csv"
filtered_data.to_csv(output_path, index=False)

print(f"✅ Saved {len(filtered_data)} matching cells to '{output_path}'")


0
✅ Saved 0 matching cells to '/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/matching_cells_4x.csv'


/tmp/ipykernel_467597/2590751370.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_data = excluded_data [duplicate_left['barcode'].isin(valid_barcodes)].copy()


In [53]:
final_data = excluded_data[~excluded_data['barcode'].isin(filtered_data['barcode'])]
print(f"no. of barcodes left {len(final_data)}")
final_data.to_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/final_data.csv",index=False)
clonotype_df.update(final_data)  # Update clonotype_df with filtered data
clonotype_df['step_number'] = 5


no. of barcodes left 6


In [54]:
df = pd.DataFrame(final_data)

# Calculate the combined UMI count (sum of alpha and beta UMIs)
df['umi_combined'] = df['umi_alpha'] + df['umi_beta']

# Now, for each barcode, assign the clonotype based on the maximum combined UMI count
# Step 1: Group by barcode to get the row with the highest combined UMI for each barcode
df_final = df.loc[df.groupby('barcode')['umi_combined'].idxmax()]

# Step 2: You may now drop the clonotype column and reassign it if needed.
# The "df_final" will have unique barcodes and the clonotype with the highest UMI count
clonotype_df.update(df_final)  # Final updates to clonotype_df
clonotype_df['step_number'] = 6 
# Show the result: barcode and its assigned clonotype_id
print(df_final[['barcode', 'clonotype_id', 'umi_combined']])


                 barcode clonotype_id  umi_combined
986   AGTGAGGCAAGCGCTC-1   clonotype1           6.0
1464  CACCAGGAGGTAGCCA-1  clonotype41          10.0
4503  TCAGCAAGTAACGCGA-1  clonotype41           6.0


In [82]:


# Final step: Ensure the most recent clonotype assignment for each barcode (most recent step_number)
final_df = clonotype_df.loc[clonotype_df.groupby('barcode')['step_number'].idxmax()]

# Ensure the number of rows remains 4515
print(f"Final DataFrame shape: {final_df.shape}")


Final DataFrame shape: (4515, 20)


In [83]:
barcode_counts_final = final_df['clonotype_id'].value_counts()
print(barcode_counts_final)

clonotype_id
clonotype54      4
clonotype7       3
clonotype1250    3
clonotype2255    3
clonotype2       3
                ..
clonotype4510    1
clonotype4511    1
clonotype4788    1
clonotype4512    1
clonotype4504    1
Name: count, Length: 4420, dtype: int64


In [84]:
import pandas as pd

# Assuming final_df is already computed and ready for updates

# Step 1: Count occurrences of clonotype_id across all barcodes
clonotype_counts = final_df['clonotype_id'].value_counts().reset_index()
clonotype_counts.columns = ['clonotype_id', 'clonotype_frequency']

# Step 2: Merge the counts back into final_df to update with clonotype frequencies
final_df = final_df.merge(clonotype_counts, on='clonotype_id', how='left')

# Step 3: Now, final_df will have a column `clonotype_frequency` with the occurrence count for each clonotype_id

# Optional: Check the updated DataFrame
print(f"Updated DataFrame shape: {final_df.shape}")

# Print the first few rows to inspect the changes
# print(final_df.head())


Updated DataFrame shape: (4515, 21)


In [86]:
final_df[["barcode","raw_clonotype_alpha","raw_clonotype_beta","combined_raw_clonotype","raw_clonotype_id","clonotype_id","clonotype_frequency"]].to_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/clonotype_DS2/t/select/final.csv",index=False)